# 13 · 🏁 The Grand Benchmark (Capstone)

Runs the **exact same** Gold aggregation (`vendas` join-broadcast
`empresas`, grouped by `setor`/`ano`/`mes`) across all 4 architectures and
compares timings. See `scripts/lab_utils.py::run_gold_benchmark`.

**You do not need every profile running at once.** Each cell below is
independent — run whichever tier's `make up-*` you currently have running,
and its result gets appended to `data/benchmark_results.json`. Come back
later with a different profile up and add to the same file. The final cell
plots whatever has accumulated so far.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "../scripts")
from lab_utils import get_connect_session, get_local_session, get_yarn_session, run_gold_benchmark

RESULTS_FILE = Path("../data/benchmark_results.json")


def save_result(result):
    results = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else []
    results = [r for r in results if r["label"] != result.label]  # replace same-label reruns
    results.append({"label": result.label, "seconds": result.seconds, "row_count": result.row_count})
    RESULTS_FILE.write_text(json.dumps(results, indent=2))
    print(f"Saved: {result}")

## Tier A — Local `local[*]`

No prerequisites — always runnable.

In [ ]:
spark = get_local_session("13-benchmark-local")
save_result(run_gold_benchmark(spark, "A — local[*]", "local"))
spark.stop()

## Tier B — Spark Standalone + Spark Connect

Requires `make up-cluster`.

In [ ]:
spark = get_connect_session("13-benchmark-connect")
save_result(run_gold_benchmark(spark, "B — Standalone+Connect", "connect"))
spark.stop()

## Tier C — YARN + HDFS

Requires `make up-hadoop`, and Lab 08's Bronze write to HDFS to have run at
least once (so `webhdfs://localhost:14000/datalake/bronze/...` exists).

In [ ]:
spark = get_yarn_session("13-benchmark-yarn")
save_result(run_gold_benchmark(spark, "C — YARN+HDFS", "hdfs"))
spark.stop()

## Tier D — Spark Standalone + S3 (RustFS)

Requires `make up-s3`, and Lab 11's Bronze write to `s3a://bronze/...` to
have run at least once.

In [ ]:
spark = get_connect_session("13-benchmark-s3")
save_result(run_gold_benchmark(spark, "D — Standalone+S3", "s3"))
spark.stop()

## The comparison dashboard

Loads whatever's accumulated in `data/benchmark_results.json` so far and
plots it — run this cell any time, even with only 1 or 2 tiers recorded.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

results = json.loads(Path("../data/benchmark_results.json").read_text())
results.sort(key=lambda r: r["label"])

labels = [r["label"] for r in results]
seconds = [r["seconds"] for r in results]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, seconds, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"][: len(labels)])
ax.set_ylabel("Seconds")
ax.set_title("Same Gold aggregation, 4 architectures")
for bar, s in zip(bars, seconds):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{s:.1f}s", ha="center", va="bottom")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Interpreting the results

A few things worth noticing (your actual numbers will vary by machine):

- **Local usually wins outright** at this dataset size — no network,
  no serialization, no cluster coordination overhead. That overhead only
  pays for itself once data outgrows a single machine (docs/05's rule of
  thumb: ~10GB).
- **HDFS and S3 timings both include a network hop** the local/shared
  -volume tiers don't pay — but S3 additionally lacks data locality
  (docs/08), which tends to show up more as dataset size grows.
- **This is the entire thesis of the lab**, made measurable: Spark's value
  isn't "faster at any size" — it's "the only option once the data no
  longer fits, or the work no longer fits, on one machine." 